# Visual-language assistant with MiniCPM-V 4.6 and OpenVINO

[MiniCPM-V 4.6](https://huggingface.co/openbmb/MiniCPM-V-4.6) is the most edge-deployment-friendly model in the MiniCPM-V series. Built on SigLIP2-400M and Qwen3.5-0.8B LLM, it inherits the strong single-image, multi-image, and video understanding capabilities of the MiniCPM-V family while significantly improving computation efficiency.

**Key Features:**

- 🔥 **Leading Foundation Capability** — Scores 13 on Artificial Analysis Intelligence Index, outperforming Qwen3.5-0.8B (10) with 19× fewer tokens.
- 💪 **Strong Multimodal Capability** — Outperforms Qwen3.5-0.8B on most vision-language tasks, reaching Qwen3.5 2B-level on OpenCompass, RefCOCO, HallusionBench, and OCRBench.
- 🚀 **Ultra-Efficient Architecture** — Based on [LLaVA-UHD v4](https://github.com/THUMAI-Lab/LLaVA-UHD-v4), reduces visual encoding FLOPs by 50%+; supports mixed 4×/16× visual token compression.
- 📱 **Broad Mobile Platform Coverage** — Deployable on iOS, Android, and HarmonyOS.

More details about the model can be found in the [model card](https://huggingface.co/openbmb/MiniCPM-V-4.6), [technical report](https://arxiv.org/abs/2509.18154), and the original [repo](https://github.com/OpenBMB/MiniCPM-V).

In this tutorial we consider how to convert and optimize MiniCPM-V 4.6 model for creating a multimodal chatbot. We use [Optimum Intel](https://github.com/huggingface/optimum-intel) for model conversion with [NNCF](https://github.com/openvinotoolkit/nncf) weight compression, and `OVModelForVisualCausalLM` for efficient inference with OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Select weight format](#Select-weight-format)
- [Prepare OpenVINO Inference Pipeline](#Prepare-OpenVINO-Inference-Pipeline)
    - [Select inference device](#Select-inference-device)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
- [Run video inference](#Run-video-inference)
- [Interactive Demo](#Interactive-Demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm-v-4.6/minicpm-v-4.6.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip uninstall -q -y transformers optimum optimum-intel optimum-onnx

%pip install -q "git+https://github.com/openvino-dev-samples/optimum-intel.git@minicpm-v4.6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers[torch]==5.7.0" "torchvision" "torchcodec" "av" "Pillow" "gradio>=4.19,<6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q nncf
%pip install -q --pre -U openvino openvino-tokenizers --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm-v-4.6.ipynb")

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

MiniCPM-V 4.6 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). For convenience, we will use OpenVINO integration with HuggingFace Optimum.

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

`optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. For int8 and int4, [NNCF](https://github.com/openvinotoolkit/nncf) will be used for weight compression.

### Select weight format
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf).

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

More details about weights compression can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [ ]:
import ipywidgets as widgets

pt_model_id = "openbmb/MiniCPM-V-4.6"

to_compress = widgets.Checkbox(
    value=True,
    description="INT4 weight compression",
    disabled=False,
)

to_compress

In [ ]:
from pathlib import Path

additional_args = {"trust-remote-code": "", "task": "image-text-to-text"}

if to_compress.value:
    model_dir = Path("MiniCPM-V-4.6-ov") / "INT4"
    additional_args.update({"weight-format": "int4", "group-size": "128", "ratio": "0.8"})
else:
    model_dir = Path("MiniCPM-V-4.6-ov") / "FP16"
    additional_args.update({"weight-format": "fp16"})

print(f"Model will be exported to: {model_dir}")

In [ ]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(pt_model_id, model_dir, additional_args=additional_args)

## Prepare OpenVINO Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

MiniCPM-V 4.6 consists of three components:

* **Vision Encoder** — SigLIP2-400M with ViT window-attention merger and MLP downsampler for extracting visual features from images.
* **Text Embeddings** — Word embeddings from the Qwen3.5 LLM.
* **Language Model** — Qwen3.5-0.8B, a hybrid SSM/Transformer architecture for text generation.

We use `OVModelForVisualCausalLM` from Optimum Intel to load all three components and run inference with OpenVINO.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

In [ ]:
from optimum.intel import OVModelForVisualCausalLM
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
ov_model = OVModelForVisualCausalLM.from_pretrained(
    model_dir,
    trust_remote_code=True,
    device=device.value,
)

print(f"Model loaded on {device.value}")

## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

The inference workflow mirrors the [model card](https://huggingface.co/openbmb/MiniCPM-V-4.6#image-inference-): we use `processor.apply_chat_template` to prepare inputs, then call `ov_model.generate`.

Key parameters:
- `downsample_mode` — `"16x"` for efficiency, `"4x"` for finer detail. **Must be passed to both `apply_chat_template` and `generate()`.**
- `max_slice_nums` — maximum image slices (recommended `36` for images, `1` for video).

In [ ]:
from PIL import Image
import requests

# Example from the MiniCPM-V-4.6 model card
example_image_url = "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/refract.png"
example_image_path = Path("refract.png")

if not example_image_path.exists():
    Image.open(requests.get(example_image_url, stream=True).raw).save(example_image_path)

image = Image.open(example_image_path).convert("RGB")
question = "What causes this phenomenon?"

display(image)
print(f"Question: {question}")

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }
]

downsample_mode = "16x"  # Using "4x" for Finer Detail

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    downsample_mode=downsample_mode,
    max_slice_nums=36,
)

print("Answer:")
output = ov_model.generate(
    **inputs,
    downsample_mode=downsample_mode,
    max_new_tokens=512,
    do_sample=False,
)

# Decode only the generated tokens (skip the prompt)
generated_ids = output[0, inputs["input_ids"].shape[1] :]
answer = processor.decode(generated_ids, skip_special_tokens=True)
print(answer)

## Run video inference
[back to top ⬆️](#Table-of-contents:)

MiniCPM-V 4.6 also supports video understanding. The snippet below mirrors the
[video inference example](https://huggingface.co/openbmb/MiniCPM-V-4.6#video-inference) from the model card.

Key parameters for video:
- `max_num_frames=128` — automatically uses uniform sampling for videos longer than 128 seconds.
- `stack_frames=1` — 1 = main frame only (recommended for short videos).
- `max_slice_nums=1` — recommended for video.
- `use_image_id=False` — disable per-frame `<image_id>` tags.
- `downsample_mode="16x"` — must also be passed to `generate()`.

> **Note:** FFmpeg or PyAV is required for video frame extraction. Install via `pip install av` or see [ffmpeg.org](https://www.ffmpeg.org/).

In [ ]:
video_messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "url": "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/football.mp4"},
            {"type": "text", "text": "Describe this video in detail."},
        ],
    }
]

downsample_mode = "16x"  # Using "4x" for Finer Detail

video_inputs = processor.apply_chat_template(
    video_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    downsample_mode=downsample_mode,
    max_num_frames=128,
    stack_frames=1,
    max_slice_nums=1,
    use_image_id=False,
)

print("Video Answer:")
video_output = ov_model.generate(
    **video_inputs,
    downsample_mode=downsample_mode,
    max_new_tokens=2048,
    do_sample=False,
)

generated_ids = video_output[0, video_inputs["input_ids"].shape[1] :]
video_answer = processor.decode(generated_ids, skip_special_tokens=True)
print(video_answer)

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Now, you can try to chat with the model. Upload an image using the `Upload` button, provide your text message into the `Input` field and click `Submit` to start communication.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, processor)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/